# 05 — Feature Engineering

**Day 2, Step 5.** Turn raw columns into model-ready features.

One rule governs this step: **every transformation lives inside the fitted
pipeline object**. Nothing is imputed, encoded or scaled outside it. That is
what makes the saved artifact self-contained and train/serve skew structurally
impossible rather than merely unlikely.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.features.engineering import FeatureEngineer, excluded_columns, feature_columns

df = load_processed("employee_attrition_processed")
print("blocked from ever becoming a feature (leakage register, F7):")
print(" ", sorted(excluded_columns()))

blocked from ever becoming a feature (leakage register, F7):
  ['Attrition', 'Employee ID', 'EmployeeNumber', 'EmployeeStatus', 'ExitDate', 'TerminationDescription', 'TerminationType', 'attrition_flag', 'employee_id', 'is_terminated', 'is_voluntary_exit', 'population']


## Engineered features — each with a reason

> *"Every engineered feature needs an actual reason — statistical or business —
> not just 'it seemed interesting'. Otherwise I'm just adding noise."*

With only 237 positive cases, noise costs recall directly.

In [3]:
for spec in get("features.engineered"):
    print(f"  {spec['name']:26} {spec['rationale']}")

  IncomePerYearAtCompany     Pay progression relative to tenure; flat pay over long tenure is a known driver.
  PromotionGap               YearsSinceLastPromotion / YearsAtCompany — stagnation signal normalised by tenure.
  SatisfactionIndex          Mean of Job/Environment/Relationship satisfaction — a single stable morale construct.
  ExperienceRatio            YearsAtCompany / TotalWorkingYears — company loyalty vs overall career mobility.
  TenureInRoleRatio          YearsInCurrentRole / YearsAtCompany — internal mobility; low mobility drives exits.
  ManagerStability           YearsWithCurrManager / YearsAtCompany — manager churn is a documented attrition driver.


In [4]:
engineered = FeatureEngineer().fit(df).transform(df)
new = [c for c in engineered.columns if c not in df.columns]
engineered[new].describe().round(3)

,IncomePerYearAtCompany,PromotionGap,SatisfactionIndex,ExperienceRatio,TenureInRoleRatio,ManagerStability
count,1470.000,1470.000,1470.000,1470.000,1470.000,1470.000
mean,1169.636,0.236,2.721,0.582,0.481,0.466
std,1353.979,0.269,0.628,0.284,0.274,0.277
min,101.571,0.000,1.000,0.000,0.000,0.000
25%,517.634,0.000,2.333,0.368,0.333,0.286
50%,793.121,0.143,2.667,0.636,0.500,0.500
75%,1217.469,0.429,3.333,0.833,0.667,0.667
max,18061.000,0.917,4.000,0.976,0.882,0.895


Note the `+1` denominators: a day-one employee produces 0 rather than
infinity. A division by zero here would surface as a NaN deep inside the model,
which is far harder to diagnose than a slightly damped ratio.

In [5]:
from hrai.features.pipeline import build_pipeline, fitted_feature_names
from sklearn.linear_model import LogisticRegression

pipeline = build_pipeline(df, LogisticRegression(max_iter=1000))
pipeline.fit(df, df["attrition_flag"])
names = fitted_feature_names(pipeline)
print(f"{len(names)} encoded features")
print("leakage check — target present?", any("Attrition" in n for n in names))

2026-08-28 01:54:25 | INFO  | pipeline built


57 encoded features
leakage check — target present? False
